# C6 example 1/4: `WELinear`

One node carries three physical 3D vectors and two scalars. C6 rotates around the z axis, so each 3D vector restricts as

$$\mathbb{R}^3\downarrow_{C_6}=E_1\oplus A,$$

where $E_1=(x,y)$ is the 2D frequency-1 irrep and $A=z$ is trivial. Therefore the input is $3E_1\oplus5A$ (three z components plus two scalar features), and the output is $E_1\oplus4A$ (one z component plus three scalars).

This notebook uses only affine intertwiners. No geometric direction or harmonics are needed because `WELinear` maps one representation-valued feature vector to another.

In [ ]:
import math
import torch
from we3nn import CyclicGroup, gspaces, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
space = gspaces.no_base_space(G)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()

# Packed layout: [v1_x,v1_y, v2_x,v2_y, v3_x,v3_y, v1_z,v2_z,v3_z, s1,s2].
input_type = nn.FieldType(space, 3 * [E1] + 5 * [A])
hidden_type = nn.FieldType(space, 2 * [regular])
# Packed layout: [out_v_x,out_v_y, out_v_z, out_s1,out_s2,out_s3].
output_type = nn.FieldType(space, [E1] + 4 * [A])
print('dimensions:', input_type.size, '->', hidden_type.size, '->', output_type.size)

In [ ]:
def pack_input(vectors, scalars):
    # vectors: (..., 3, 3); scalars: (..., 2)
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    z = vectors[..., :, 2]
    return torch.cat((xy, z, scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    z = x[..., 6:9]
    return torch.cat((xy, z.unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    vector = torch.cat((y[..., :2], y[..., 2:3]), dim=-1)
    return vector, y[..., 3:6]

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
x = input_type.wrap(pack_input(vectors, scalars))
print('packed input:', x.tensor)

## Architecture

The two learnable maps are complete bases of $C_6$ intertwiners:

$$x\xrightarrow{W_1}2\,\mathrm{Reg}_{C_6}\xrightarrow{\mathrm{ReLU}}2\,\mathrm{Reg}_{C_6}\xrightarrow{W_2}y.$$

A regular representation is a permutation representation, so coordinatewise ReLU (`PointActiv`) commutes with the group action.

In [ ]:
class WELinearNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.input_layer = nn.WELinear(input_type, hidden_type)
        self.activation = nn.PointActiv(hidden_type, torch.relu)
        self.output_layer = nn.WELinear(hidden_type, output_type)

    def forward(self, features):
        h_pre = self.input_layer(features)
        h = self.activation(h_pre)
        return self.output_layer(h), h_pre, h

model = WELinearNetwork().eval()
y, h_pre, h = model(x)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('packed output:', y.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))

## All six rotations and an explicit equivariance test

For each $g=r^k$, we compute the network again on $\rho_{in}(g)x$ and compare it with $\rho_{out}(g)f(x)$. The printed input/output vectors are reconstructed as ordinary 3D vectors; scalar columns are unchanged by C6.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    y_k, _, _ = model(x_k)
    expected_k = y.transform_fibers(element)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=2e-5, rtol=2e-5)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  input vectors :', in_vectors_k[0].tolist())
    print('  input scalars :', in_scalars_k[0].tolist())
    print('  output vector :', out_vector_k[0].tolist())
    print('  output scalars:', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))